# Deployments, ReplicaSets & Rollouts

## What's covered

- **Why bare pods aren't enough** — controllers, reconciliation, and the difference between "a pod exists" and "a workload is running"
- **ReplicaSet** — the controller that keeps N copies of a pod alive against a label selector
- **Deployment** — the controller that manages ReplicaSets so you can ship new versions safely
- **Anatomy of a Deployment manifest** — `spec.replicas`, `spec.selector`, `spec.template`, and how they fit together
- **The ownership chain** — Deployment owns ReplicaSet owns Pods, traced through `metadata.ownerReferences`
- **Scaling** — `kubectl scale`, the `replicas` field, and what a controller does when the number changes
- **Rolling updates** — `maxSurge`, `maxUnavailable`, and how Kubernetes ships a new version without dropping traffic
- **Rollout history and rollback** — `kubectl rollout history`, `kubectl rollout undo`, the `change-cause` annotation
- **Update strategies** — `RollingUpdate` versus `Recreate`, and when to pick each
- **The rest of the workload family** — `DaemonSet`, `StatefulSet`, `Job`, `CronJob` — what each one's for and where the course revisits them

By the end of this notebook you should be able to write a Deployment manifest, ship a new image with zero downtime, roll it back if it misbehaves, and recognise which workload kind each problem calls for. Notebooks 01 and 02 are the prerequisites — you should already be comfortable with the object model and label selectors, because controllers live or die by them.

## Why bare pods aren't enough

Notebook 02 created pods directly. That works for learning, but it falls over in production for two reasons.

**Bare pods don't survive node failure.** A `Pod` object lives in `etcd` referencing a specific node. If that node dies, the kubelet there stops reporting in. The pod's phase eventually becomes `Unknown`, and *nothing* moves it elsewhere. The pod is just gone.

**Bare pods don't roll forward.** Suppose you've deployed `web` running `nginx:1.27-alpine` and you want to update to `1.28`. With bare pods, you'd need to delete the old pod and create a new one — there's a window with zero pods running. For a long-running service, that's a service outage.

What you really want is a **higher-level object** that says: "I want three copies of *this pod template* alive at all times, anywhere in the cluster, and when I change the template, replace the running pods gracefully." That object is the **Deployment**.

### Controllers — the missing word

A **controller** is a program (usually inside `kube-controller-manager`) that watches a particular kind of object and works to make reality match its `spec`. You've already met one without knowing it: the kubelet is a controller for pods on its node. There are many others:

- **ReplicaSet controller** — for each ReplicaSet, ensures the right number of matching pods exist.
- **Deployment controller** — for each Deployment, manages a ReplicaSet (and creates new ReplicaSets during rollouts).
- **DaemonSet controller** — ensures one pod per node.
- **StatefulSet controller** — manages pods with stable identities and storage.
- **Job controller** — runs a pod to completion.
- **Node controller** — marks nodes unreachable when they stop heart-beating.

Every controller is the same loop: read the desired state from `etcd`, observe reality, take action to close the gap, write the observed state back to `status`. That's the whole game. We met this pattern in notebook 02; now we'll see what it actually produces.

## ReplicaSet — keeping N pods alive

The **ReplicaSet** is the simplest workload controller. Its job: keep exactly `N` pods alive that match a label selector.

```yaml
apiVersion: apps/v1
kind: ReplicaSet
metadata:
  name: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web         # MUST match the selector
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
          ports:
            - containerPort: 80
```

Three top-level fields in `spec`:

- **`replicas`** — the target count. "How many pods I want."
- **`selector`** — the label query used to identify "my pods."
- **`template`** — a complete pod spec. Everything below `spec.template.spec` is exactly what you'd put in a Pod manifest.

When the ReplicaSet controller observes this object, it runs a tight loop:

1. Count the pods matching `selector.matchLabels`.
2. If fewer than `replicas` — create new pods from `template`.
3. If more than `replicas` — delete the excess.
4. Repeat forever.

**The selector and the template labels must agree.** If your `selector.matchLabels` says `app: web` but `template.metadata.labels` says `app: api`, the controller will *create pods that it then can't see* — an infinite loop of "I need three pods, I see zero, let me make three more." Kubernetes catches this at admission time and rejects the manifest. Still, it's a real footgun in custom controllers.

**Pods created by a ReplicaSet get a name like `<rs-name>-<random>`.** Their `metadata.ownerReferences` points back to the ReplicaSet — that's how the cluster says "this pod belongs to that controller, delete it when the parent is deleted."

### The self-healing demo

If we delete a pod that a ReplicaSet owns, the controller notices within seconds and creates a replacement — same template, new name. We'll see this below.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: ReplicaSet
metadata:
  name: web-rs
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web-rs
  template:
    metadata:
      labels:
        app: web-rs
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
EOF
!sleep 6
# Three pods, all owned by the ReplicaSet
!kubectl get rs web-rs
!echo '---'
!kubectl get pods -l app=web-rs
!echo '---'
# Delete one — watch the RS create a replacement
!POD=$(kubectl get pods -l app=web-rs -o jsonpath='{.items[0].metadata.name}') && echo "deleting $POD" && kubectl delete pod $POD
!sleep 4
!kubectl get pods -l app=web-rs

## Why you almost never write a ReplicaSet directly

ReplicaSets are great at *keeping N pods alive*. They are terrible at *changing what those pods run*. Try to update the image on a ReplicaSet and you'll find:

- Editing `template.spec.containers[0].image` on an existing ReplicaSet **does not restart any pods.** The new image only applies to pods *created from now on*. Existing pods keep running the old image.
- To actually roll the change out, you'd have to delete pods one by one and let the controller recreate them. With three pods that's annoying; with thirty, error-prone.
- There's no history. Once you've changed the template, the old one is gone — no way to "go back to last week's version" without finding the manifest somewhere.

What you really want is a controller that owns *multiple* ReplicaSets — one per version — and choreographs the transition from one to the next, with history and a one-command rollback. That's the **Deployment**.

```
+------------------+
|   Deployment     |  you write this
|   (version 2)    |
+--------+---------+
         | owns
         v
+------------------+     +------------------+
| ReplicaSet (v2)  |     | ReplicaSet (v1)  |
|   replicas: 3    |     |   replicas: 0    |  scaled down by the Deployment controller
+--------+---------+     +------------------+
         | owns
         v
   +-----+-----+
   |     |     |
   v     v     v
  pod   pod   pod
```

You write the Deployment. The **Deployment controller** creates a ReplicaSet for the current version. The **ReplicaSet controller** creates the pods. When you change the Deployment's template, the Deployment controller creates a *new* ReplicaSet and gradually scales it up while scaling the old one down. The old ReplicaSet sticks around at `replicas: 0` so you can roll back to it.

This three-layer ownership — Deployment owns ReplicaSets owns Pods — is one of the most-tested topics on the CKA. Memorise the picture.

## Anatomy of a Deployment manifest

A Deployment looks almost identical to a ReplicaSet — same `replicas`, `selector`, `template` — plus a few fields that govern *how the controller updates pods*.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  labels:
    app: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 25%        # or an integer
      maxUnavailable: 25%  # or an integer
  revisionHistoryLimit: 10
  minReadySeconds: 0
  progressDeadlineSeconds: 600
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
          ports:
            - containerPort: 80
```

Field by field:

**`spec.replicas`** — the same target count you saw on ReplicaSet. The Deployment passes this through to the active ReplicaSet.

**`spec.selector.matchLabels`** — must match `template.metadata.labels`. **Once set, the selector is immutable on a Deployment** — you can't change it after creation. This avoids the orphan-pods problem.

**`spec.template`** — the pod template. This is the field you'll change most often (typically to bump an image). Any change here triggers a rollout.

**`spec.strategy.type`** — `RollingUpdate` (the default, gradual replacement) or `Recreate` (kill all, then start all). Covered below.

**`spec.strategy.rollingUpdate.maxSurge`** — how many *extra* pods the controller is allowed to create *above* `replicas` during a rollout. `25%` of 4 replicas means 1 extra pod.

**`spec.strategy.rollingUpdate.maxUnavailable`** — how many of the desired pods can be unavailable *at the same time* during a rollout. `25%` of 4 means at most 1 down.

**`spec.revisionHistoryLimit`** — how many old ReplicaSets to keep around for rollback. Defaults to 10.

**`spec.minReadySeconds`** — after a new pod is `Ready`, wait this long before treating it as "available." Useful when readiness probes are lenient but you want extra confidence.

**`spec.progressDeadlineSeconds`** — if a rollout doesn't make progress within this long, mark it failed.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  labels:
    app: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
          ports:
            - containerPort: 80
EOF
!sleep 8
# Three-layer ownership: Deployment -> ReplicaSet -> Pods
!kubectl get deploy,rs,pods -l app=web
!echo '---'
# Read each ReplicaSet's owner — it points back to the Deployment
!kubectl get rs -l app=web -o jsonpath='{range .items[*]}{.metadata.name}{"  owner="}{.metadata.ownerReferences[0].kind}{"/"}{.metadata.ownerReferences[0].name}{"\n"}{end}'

## Scaling — changing the desired count

Scaling a Deployment up or down is a one-field change. The controller does the rest.

```bash
# Imperative, fastest for the exam
kubectl scale deployment web --replicas=5

# Declarative — edit the manifest and reapply
kubectl edit deployment web   # or change spec.replicas in your YAML and `kubectl apply -f`
```

When you change `replicas`:

- **Scale up.** The Deployment controller updates the active ReplicaSet's `replicas`. The ReplicaSet controller sees more desired than actual and creates the difference. Each new pod goes through `Pending` → `ContainerCreating` → `Running` → `Ready`.
- **Scale down.** The ReplicaSet controller sees more actual than desired. It picks pods to delete — preferring `NotReady` pods, then pods on nodes that already have more replicas of the set, then by creation time. The selected pods receive `SIGTERM`, get `terminationGracePeriodSeconds` to exit, then `SIGKILL`.

Scaling does **not** create a new ReplicaSet. Same generation, same template, just a different count — which is why scaling is not considered a "rollout" and won't show up in `kubectl rollout history`.

### Horizontal Pod Autoscaler — a glance

`replicas` is the manual lever. You can also delegate it to a **HorizontalPodAutoscaler** (HPA) — an object that watches a metric (CPU, memory, or anything from the metrics API) and updates the Deployment's `replicas` on your behalf:

```bash
kubectl autoscale deployment web --min=2 --max=10 --cpu-percent=70
```

The HPA controller polls the metric every 15 seconds and adjusts `replicas` to keep CPU near 70%. We'll come back to autoscaling properly when we cover resources in notebook 07.

In [ ]:
# Scale up
!kubectl scale deployment web --replicas=5
!sleep 5
!kubectl get deploy web
!kubectl get pods -l app=web --no-headers | wc -l
!echo '---'
# Scale back down
!kubectl scale deployment web --replicas=3
!sleep 5
!kubectl get deploy web
!kubectl get pods -l app=web --no-headers | wc -l

## Rolling updates — shipping a new version with no downtime

The most-used Deployment feature is the rollout. You change something — almost always the container image — and Kubernetes replaces pods one batch at a time, keeping at least `replicas - maxUnavailable` of them serving traffic the whole time.

```bash
# Imperative: change the image directly
kubectl set image deployment/web app=nginx:1.28-alpine

# Or: edit the manifest in place and apply
```

What happens under the hood when the image changes:

1. The Deployment controller notices `spec.template` has a new hash.
2. It creates a **new ReplicaSet** for the new template, starting at `replicas: 0`.
3. It scales the new ReplicaSet up by `maxSurge`, while scaling the old ReplicaSet down by `maxUnavailable`, *in lockstep*. With the default 25% / 25% and 4 replicas, that means: new RS to 1, old to 3, wait for the new pod's `Ready`, new to 2, old to 2, and so on.
4. As pods on the new ReplicaSet become `Ready`, they take traffic — Services route by selector (notebook 04).
5. When the new ReplicaSet has `replicas` matching the spec and the old has `0`, the rollout is complete.
6. The old ReplicaSet stays around for rollback. Its template is preserved exactly, so `undo` can recreate the previous state.

### `maxSurge` and `maxUnavailable` — the trade-off

These two knobs control the speed-vs-safety trade-off of a rollout:

| Setting | Behaviour |
|---|---|
| `maxSurge: 0, maxUnavailable: 1` | Strictly take one pod down before bringing one up. Conservative; fits when you can't spare extra capacity. |
| `maxSurge: 1, maxUnavailable: 0` | Strictly bring one extra pod up before taking any old one down. The "always at full capacity" choice. |
| `maxSurge: 25%, maxUnavailable: 25%` *(default)* | Fast — overlap surge and shutdown. Saves time but briefly uses extra capacity *and* runs slightly under desired. |

Both can be either an absolute integer or a percentage of `replicas`. They can't both be zero — that would mean "no progress allowed."

### Watching a rollout

```bash
kubectl rollout status deployment/web   # blocks until rollout finishes or fails
kubectl get pods -l app=web -w          # watch pods come and go
kubectl describe deployment web         # events at the bottom show the rollout step-by-step
```

In [ ]:
# Trigger a rollout by changing the image
!kubectl set image deployment/web app=nginx:1.28-alpine
!echo '---'
# Block until the rollout is done (or fails)
!kubectl rollout status deployment/web --timeout=120s
!echo '---'
# Two ReplicaSets now exist — old at 0, new at 3
!kubectl get rs -l app=web

## Rollout history and rollback

Each rollout creates a new ReplicaSet, and the Deployment keeps a numbered history.

```bash
kubectl rollout history deployment/web
```

Output looks like:

```
deployment.apps/web
REVISION  CHANGE-CAUSE
1         <none>
2         <none>
```

To see the spec of a specific revision:

```bash
kubectl rollout history deployment/web --revision=2
```

### Roll back

If revision 2 is misbehaving, one command takes you back:

```bash
kubectl rollout undo deployment/web                 # back to the immediately previous revision
kubectl rollout undo deployment/web --to-revision=1 # back to a specific revision
```

`undo` is just another rollout — it creates a *new* revision whose template equals the old one, then runs the same rolling-update process to switch to it. So after `undo` you'll see revisions 1, 2, 3 — where 3 has the same template as 1.

### Pausing and resuming

For risky rollouts you can pause mid-flight, inspect the partial state, then resume:

```bash
kubectl rollout pause deployment/web    # any further template edits won't trigger a rollout
# ... batch a few more changes ...
kubectl rollout resume deployment/web   # all the queued changes ship as one rollout
```

### The `change-cause` annotation

The `CHANGE-CAUSE` column above is read from the `kubernetes.io/change-cause` annotation. Set it when you ship to leave breadcrumbs for future-you:

```bash
kubectl annotate deployment/web kubernetes.io/change-cause="bump nginx to 1.28 for CVE-2025-xxxx"
```

Anything that triggers a new revision *after* the annotation is set inherits that change-cause.

In [ ]:
!kubectl rollout history deployment/web
!echo '---'
# Roll back to the previous revision
!kubectl rollout undo deployment/web
!kubectl rollout status deployment/web --timeout=120s
!echo '---'
# History now shows a new revision; image is back to 1.27
!kubectl rollout history deployment/web
!kubectl get deploy web -o jsonpath='Current image: {.spec.template.spec.containers[0].image}{"\n"}'

## Update strategies — `RollingUpdate` vs `Recreate`

A Deployment's `spec.strategy.type` chooses between two algorithms.

**`RollingUpdate`** (default). What we just walked through. The new ReplicaSet scales up while the old one scales down, governed by `maxSurge` / `maxUnavailable`. **No downtime** if your app handles `SIGTERM` cleanly and your readiness probe is honest.

**`Recreate`.** The old ReplicaSet is scaled to 0 first, then the new one is scaled up. **There will be a window with zero pods.** Use this when:

- The new version can't coexist with the old (incompatible database schema migrations, conflicting filesystem state).
- You're using a `ReadWriteOnce` PersistentVolume that only one pod can mount at a time (notebook 06).
- You're using a `HostPort` and can't have two pods on the same node briefly competing for it.

`Recreate` is the simpler algorithm; `RollingUpdate` is the production default. The CKA expects you to know both names and the *kind* of situation that argues for `Recreate`.

### What triggers a rollout

A rollout fires whenever `spec.template` *changes*. Specifically:

- Changing a container image, env var, command, args, volume mount, resource, probe, security context — anything inside `template`.
- Changing the labels on the pod template (carefully — your selector and template labels still have to match).
- Adding or removing a container, init container, or volume.

A rollout **does not** fire when:

- You change `spec.replicas` (that's a scale, not a rollout).
- You change Deployment-level fields outside `spec.template` (like `strategy`, `revisionHistoryLimit`, `progressDeadlineSeconds`).
- You change `metadata.labels` on the Deployment itself (not on the template).

If you ever want to force a rollout without changing the template — for example to make pods pick up a new ConfigMap value (notebook 05) — the canonical trick is to bump an annotation in the template:

```bash
kubectl patch deployment web -p \
  '{"spec":{"template":{"metadata":{"annotations":{"rolled-at":"'$(date +%s)'"}}}}}'
```

## The rest of the workload family

Deployments cover stateless services — the most common case. But Kubernetes ships four other workload controllers, each for a specific shape of workload. We'll meet them properly later in the course; here's the menu.

### DaemonSet

> *One pod per node.*

The DaemonSet controller ensures **exactly one** copy of a pod runs on every node that matches a selector. Add a node — a pod appears on it. Remove a node — its pod is gone. Use it for **node-level agents**:

- log shippers (Fluent Bit, Vector)
- metrics agents (`node-exporter`, the Datadog agent)
- networking and storage plugins (the CNI agent, the CSI node driver)
- security agents (Falco)

You'll see DaemonSets in the wild in `kube-system` — `kube-proxy` and the CNI plugin both run as DaemonSets.

### StatefulSet

> *Pods with identity and storage that survive restarts.*

A Deployment treats pods as interchangeable. A **StatefulSet** treats pods as ordinals — `pod-0`, `pod-1`, `pod-2` — each with a **stable name**, a **stable network identity**, and **its own PersistentVolume that follows it across restarts**. Use it for **stateful systems** where the identity matters:

- databases (PostgreSQL, MySQL primary/replica setups)
- distributed datastores (Cassandra, Elasticsearch, Kafka brokers)
- consensus systems (etcd, Zookeeper)

The StatefulSet's network identity is delivered through a *headless Service* (notebook 04) and its storage through *volumeClaimTemplates* (notebook 06).

### Job

> *Run a pod to completion.*

A **Job** creates one or more pods and watches them until they exit successfully. If a pod fails, the Job retries (subject to `backoffLimit`). Use it for **batch work**:

- one-off database migrations
- nightly report generation
- ETL pipelines

A Job's pod template defaults to `restartPolicy: OnFailure`. Set `spec.parallelism` to run multiple pods in parallel, and `spec.completions` for "I want N successful pods before declaring victory."

### CronJob

> *A Job, on a schedule.*

A **CronJob** is exactly what it sounds like — a Job templated to fire on a cron schedule:

```yaml
spec:
  schedule: "0 2 * * *"   # standard cron syntax
  jobTemplate:
    spec:
      template:
        spec:
          ...pod spec...
```

Use it for **periodic batch work**: backups, log rotation, cache warmers, recurring reports. CronJobs handle missed runs, concurrent runs, and history cleanup — knobs we'll meet alongside Jobs later.

### Picking the right controller

A small flowchart in your head:

```
        Does the workload run forever?
        /                            \
      Yes                             No
       |                               |
  Needs stable identity      One-shot or scheduled?
  and per-pod storage?         /             \
     /         \           Once         On a schedule
   Yes          No           |                  |
    |           |           Job             CronJob
StatefulSet  Needs one
             copy per node?
             /        \
           Yes         No
            |           |
       DaemonSet   Deployment
```

For the CKA, *Deployment* is the answer most of the time. Reach for the others only when the question describes the specific shape they fit.

In [ ]:
# kube-proxy and the CNI plugin are DaemonSets — one per node
!kubectl get daemonsets -n kube-system
!echo '---'
# Their pods, one per node
!kubectl get pods -n kube-system -o wide --no-headers | awk '{print $1, $7}' | head -10

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 04:

```bash
kubectl delete deployment web
kubectl delete replicaset web-rs
```

Or, more broadly, every Deployment and ReplicaSet in the `default` namespace:

```bash
kubectl delete deployment,replicaset --all
```

Note: deleting a Deployment deletes its ReplicaSets, which deletes their pods — `ownerReferences` cascade by default. So one `kubectl delete deployment web` cleans the whole tree.